# 04 — Master Merge
Joins all four federal datasets into a single analysis-ready parquet file.

**Input:** flights (20.6M) + NOAA weather (100 airports) + BTS Form 41 financials + BTS T-100 airport traffic

**Output:** `dataset/merged_flights.parquet` — 18.2M rows × 77 cols

**Match rates:** Weather 91.4% · Financials 100% · T-100 100%

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import gc

In [2]:
ROOT     = Path("dataset")
OUT_PATH = ROOT / "merged_flights.parquet"
 
FLIGHTS_PATH    = ROOT / "data.parquet"
WEATHER_PATH    = ROOT / "weather" / "weather_clean.parquet"
FINANCIALS_PATH = ROOT / "financials" / "financials_clean.parquet"
T100_PATH       = ROOT / "airports" / "t100_clean.parquet"

In [3]:
def shape_report(df: pd.DataFrame, label: str) -> None:
    print(f"  [{label}] {df.shape[0]:,} rows × {df.shape[1]} cols")
 
 
def match_rate(df: pd.DataFrame, null_col: str, label: str) -> None:
    matched = df[null_col].notna().sum()
    pct     = matched / len(df) * 100
    print(f"  [{label}] matched {matched:,} / {len(df):,} rows ({pct:.1f}%)")

## Load All Datasets

In [4]:
print("\n── 1. Loading datasets ──────────────────────────────────────────────")
 
flights = pd.read_parquet(FLIGHTS_PATH)
shape_report(flights, "flights raw")
 
weather = pd.read_parquet(WEATHER_PATH)
shape_report(weather, "weather raw")
 
financials = pd.read_parquet(FINANCIALS_PATH)
shape_report(financials, "financials raw")
 
t100 = pd.read_parquet(T100_PATH)
shape_report(t100, "t100 raw")


── 1. Loading datasets ──────────────────────────────────────────────
  [flights raw] 20,588,154 rows × 46 cols
  [weather raw] 2,322,436 rows × 14 cols
  [financials raw] 165 rows × 15 cols
  [t100 raw] 755,277 rows × 36 cols


In [5]:
if not pd.api.types.is_datetime64_any_dtype(flights["FL_DATE"]):
    flights["FL_DATE"] = pd.to_datetime(flights["FL_DATE"])
 
print(f"  flights  : {flights['FL_DATE'].min().date()} → {flights['FL_DATE'].max().date()}")
 
if not pd.api.types.is_datetime64_any_dtype(weather["DATETIME"]):
    weather["DATETIME"] = pd.to_datetime(weather["DATETIME"])
 
print(f"  weather  : {weather['DATETIME'].min().date()} → {weather['DATETIME'].max().date()}")
 
WEATHER_END = weather["DATETIME"].max().normalize()
print(f"\n  ✂  Trimming flights to weather window: ≤ {WEATHER_END.date()}")
 
flights = flights[flights["FL_DATE"] <= WEATHER_END].copy()
shape_report(flights, "flights trimmed")

  flights  : 2023-01-01 → 2025-12-31
  weather  : 2023-01-01 → 2025-08-27

  ✂  Trimming flights to weather window: ≤ 2025-08-27
  [flights trimmed] 18,227,796 rows × 46 cols


In [6]:
print("── FINANCIALS columns & sample ──────────────────────────────────────")
print(financials.columns.tolist())
print(financials.head(3))

print("\n── T-100 columns & sample ───────────────────────────────────────────")
print(t100.columns.tolist())
print(t100.head(3))

print("\n── FLIGHTS carrier column ───────────────────────────────────────────")
print(flights["OP_UNIQUE_CARRIER"].unique()[:10])

print("\n── FLIGHTS airport columns ──────────────────────────────────────────")
print(flights["ORIGIN"].unique()[:10])

── FINANCIALS columns & sample ──────────────────────────────────────
['NET_INCOME', 'OP_PROFIT_LOSS', 'TRANS_REV_PAX', 'OP_REVENUES', 'FLYING_OPS', 'MAINTENANCE', 'PAX_SERVICE', 'DEPREC_AMORT', 'TRANS_EXPENSES', 'OP_EXPENSES', 'UNIQUE_CARRIER', 'UNIQUE_CARRIER_NAME', 'REGION', 'YEAR', 'QUARTER']
   NET_INCOME  OP_PROFIT_LOSS  TRANS_REV_PAX  OP_REVENUES  FLYING_OPS  \
0  -539748.67      -117804.23     1333272.28   1558333.44   799886.03   
1  -520092.14      -398791.62      543730.03   1038862.98   611263.80   
2  -406248.43      -500721.12     1259141.55   1501303.86   927668.65   

   MAINTENANCE  PAX_SERVICE  DEPREC_AMORT  TRANS_EXPENSES  OP_EXPENSES  \
0    157381.39    159208.60     114840.58          392.83   1676137.68   
1     75788.40     74522.95      66966.33            0.00   1437654.60   
2    202931.39    193397.61     140276.33          414.51   2002024.98   

  UNIQUE_CARRIER UNIQUE_CARRIER_NAME REGION  YEAR  QUARTER  
0             B6     JetBlue Airways      D  2024  

In [7]:
print("\n── 3. Building join keys on flights ─────────────────────────────────")

flights["YEAR"]    = flights["FL_DATE"].dt.year.astype("int16")
flights["MONTH"]   = flights["FL_DATE"].dt.month.astype("int8")
flights["QUARTER"] = flights["FL_DATE"].dt.quarter.astype("int8")

flights["DEP_HOUR"] = (flights["CRS_DEP_TIME"] // 100).clip(0, 23).astype("int8")
flights["ARR_HOUR"] = (flights["CRS_ARR_TIME"] // 100).clip(0, 23).astype("int8")

flights["FL_DATE_DATE"] = flights["FL_DATE"].dt.date

print("  Keys added: YEAR, MONTH, QUARTER, DEP_HOUR, ARR_HOUR, FL_DATE_DATE")

print(f"  DEP_HOUR range: {flights['DEP_HOUR'].min()} → {flights['DEP_HOUR'].max()}")
print(f"  ARR_HOUR range: {flights['ARR_HOUR'].min()} → {flights['ARR_HOUR'].max()}")
print(f"  YEAR values: {sorted(flights['YEAR'].unique())}")
print(f"  QUARTER values: {sorted(flights['QUARTER'].unique())}")


── 3. Building join keys on flights ─────────────────────────────────
  Keys added: YEAR, MONTH, QUARTER, DEP_HOUR, ARR_HOUR, FL_DATE_DATE
  DEP_HOUR range: 0 → 23
  ARR_HOUR range: 0 → 23
  YEAR values: [2023, 2024, 2025]
  QUARTER values: [1, 2, 3, 4]


## Weather Join — Origin Airport
Matches hourly NOAA weather to each flight's departure airport and hour.

In [8]:
print("\n── 4. Weather join — ORIGIN ─────────────────────────────────────────")

if not pd.api.types.is_datetime64_any_dtype(weather["DATETIME"]):
    weather["DATETIME"] = pd.to_datetime(weather["DATETIME"])

weather["WX_DATE"] = weather["DATETIME"].dt.date
weather["WX_HOUR"] = weather["DATETIME"].dt.hour.astype("int8")

WX_COLS = [
    "AIRPORT", "WX_DATE", "WX_HOUR",
    "TEMP", "DEW_POINT", "PRESSURE",
    "WIND_DIR", "WIND_SPEED", "SKY_CONDITION",
    "PRECIP_1HR", "PRECIP_6HR",
]
wx_cols_present = [c for c in WX_COLS if c in weather.columns]
print(f"  Weather cols available: {wx_cols_present}")
wx = weather[wx_cols_present].copy()

wx_origin = wx.rename(columns={
    "AIRPORT"      : "ORIGIN",
    "WX_DATE"      : "FL_DATE_DATE",
    "WX_HOUR"      : "DEP_HOUR",
    "TEMP"         : "origin_temp",
    "DEW_POINT"    : "origin_dew_point",
    "PRESSURE"     : "origin_pressure",
    "WIND_DIR"     : "origin_wind_dir",
    "WIND_SPEED"   : "origin_wind_speed",
    "SKY_CONDITION": "origin_sky_condition",
    "PRECIP_1HR"   : "origin_precip_1hr",
    "PRECIP_6HR"   : "origin_precip_6hr",
})

flights = flights.merge(wx_origin, on=["ORIGIN", "FL_DATE_DATE", "DEP_HOUR"], how="left")

matched = flights["origin_temp"].notna().sum()
pct = matched / len(flights) * 100
print(f"  Matched {matched:,} / {len(flights):,} rows ({pct:.1f}%)")
print(f"  New shape: {flights.shape}")


── 4. Weather join — ORIGIN ─────────────────────────────────────────
  Weather cols available: ['AIRPORT', 'WX_DATE', 'WX_HOUR', 'TEMP', 'DEW_POINT', 'PRESSURE', 'WIND_DIR', 'WIND_SPEED', 'SKY_CONDITION', 'PRECIP_1HR', 'PRECIP_6HR']
  Matched 16,657,573 / 18,227,796 rows (91.4%)
  New shape: (18227796, 57)


## Weather Join — Destination Airport
Matches weather to destination airport using estimated arrival hour.

In [9]:
print("\n── 5. Weather join — DESTINATION ────────────────────────────────────")

wx_dest = wx.rename(columns={
    "AIRPORT"      : "DEST",
    "WX_DATE"      : "FL_DATE_DATE",
    "WX_HOUR"      : "ARR_HOUR",
    "TEMP"         : "dest_temp",
    "DEW_POINT"    : "dest_dew_point",
    "PRESSURE"     : "dest_pressure",
    "WIND_DIR"     : "dest_wind_dir",
    "WIND_SPEED"   : "dest_wind_speed",
    "SKY_CONDITION": "dest_sky_condition",
    "PRECIP_1HR"   : "dest_precip_1hr",
    "PRECIP_6HR"   : "dest_precip_6hr",
})

flights = flights.merge(wx_dest, on=["DEST", "FL_DATE_DATE", "ARR_HOUR"], how="left")

matched = flights["dest_temp"].notna().sum()
pct = matched / len(flights) * 100
print(f"  Matched {matched:,} / {len(flights):,} rows ({pct:.1f}%)")
print(f"  New shape: {flights.shape}")

del wx, wx_origin, wx_dest, weather
gc.collect()
print("  Weather dataframes freed from memory.")


── 5. Weather join — DESTINATION ────────────────────────────────────
  Matched 16,649,761 / 18,227,796 rows (91.3%)
  New shape: (18227796, 65)
  Weather dataframes freed from memory.


## Financials Join
Merges BTS Form 41 quarterly financials on carrier + year + quarter.

In [10]:
print("\n── 6. Financials join ───────────────────────────────────────────────")

FIN_KEEP = [
    "UNIQUE_CARRIER", "YEAR", "QUARTER",
    "OP_REVENUES", "OP_EXPENSES", "NET_INCOME", "OP_PROFIT_LOSS",
    "FLYING_OPS", "MAINTENANCE", "PAX_SERVICE",
    "TRANS_REV_PAX", "DEPREC_AMORT", "TRANS_EXPENSES",
]
fin = financials[FIN_KEEP].copy()
fin = fin.rename(columns={"UNIQUE_CARRIER": "OP_UNIQUE_CARRIER"})
fin["YEAR"]    = fin["YEAR"].astype("int16")
fin["QUARTER"] = fin["QUARTER"].astype("int8")

flights = flights.merge(fin, on=["OP_UNIQUE_CARRIER", "YEAR", "QUARTER"], how="left")

matched = flights["OP_REVENUES"].notna().sum()
pct = matched / len(flights) * 100
print(f"  Matched {matched:,} / {len(flights):,} rows ({pct:.1f}%)")
print(f"  New shape: {flights.shape}")

del fin, financials
gc.collect()
print("  Financials freed from memory.")


── 6. Financials join ───────────────────────────────────────────────
  Matched 18,227,796 / 18,227,796 rows (100.0%)
  New shape: (18227796, 75)
  Financials freed from memory.


## T-100 Airport Traffic Join
Merges monthly passenger volumes to derive `origin_monthly_passengers`.

In [11]:
print("\n── 7. T-100 airport join ────────────────────────────────────────────")

flight_carriers = set(flights["OP_UNIQUE_CARRIER"].unique())
t100_filtered = t100[t100["UNIQUE_CARRIER"].isin(flight_carriers)].copy()
print(f"  T-100 after carrier filter: {len(t100_filtered):,} rows (from {len(t100):,})")

t100_agg = (
    t100_filtered
    .groupby(["ORIGIN", "YEAR", "MONTH"], as_index=False)
    .agg(
        origin_monthly_passengers=("PASSENGERS", "sum"),
        origin_monthly_departures=("PASSENGERS", "count"),
    )
)
t100_agg["YEAR"]  = t100_agg["YEAR"].astype("int16")
t100_agg["MONTH"] = t100_agg["MONTH"].astype("int8")

print(f"  T-100 aggregated: {t100_agg.shape}")

flights = flights.merge(t100_agg, on=["ORIGIN", "YEAR", "MONTH"], how="left")

matched = flights["origin_monthly_passengers"].notna().sum()
pct = matched / len(flights) * 100
print(f"  Matched {matched:,} / {len(flights):,} rows ({pct:.1f}%)")
print(f"  New shape: {flights.shape}")

del t100, t100_filtered, t100_agg
gc.collect()
print("  T-100 freed from memory.")


── 7. T-100 airport join ────────────────────────────────────────────
  T-100 after carrier filter: 459,568 rows (from 755,277)
  T-100 aggregated: (13010, 5)
  Matched 18,227,711 / 18,227,796 rows (100.0%)
  New shape: (18227796, 77)
  T-100 freed from memory.


## Final Validation

In [12]:
print("\n── 8. Final validation ──────────────────────────────────────────────")

print(f"  Final shape: {flights.shape}")
print(f"\n  All columns:")
for col in flights.columns:
    print(f"    {col}: {flights[col].dtype}")

print(f"\n  Null summary (cols with any nulls):")
null_counts = flights.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
for col, cnt in null_counts.items():
    pct = cnt / len(flights) * 100
    print(f"    {col}: {cnt:,} ({pct:.1f}%)")

print(f"\n  Saving → {OUT_PATH}")
flights.to_parquet(OUT_PATH, index=False, engine="pyarrow", compression="snappy")

import os
size_mb = os.path.getsize(OUT_PATH) / 1024 / 1024
print(f"  ✅ Saved. File size: {size_mb:.1f} MB")


── 8. Final validation ──────────────────────────────────────────────
  Final shape: (18227796, 77)

  All columns:
    YEAR: int16
    QUARTER: int8
    MONTH: int8
    DAY_OF_MONTH: int64
    DAY_OF_WEEK: int64
    FL_DATE: datetime64[ns]
    OP_UNIQUE_CARRIER: object
    OP_CARRIER_AIRLINE_ID: int64
    TAIL_NUM: object
    OP_CARRIER_FL_NUM: float64
    ORIGIN: object
    ORIGIN_CITY_NAME: object
    ORIGIN_STATE_ABR: object
    ORIGIN_STATE_NM: object
    DEST: object
    DEST_CITY_NAME: object
    DEST_STATE_ABR: object
    DEST_STATE_NM: object
    CRS_DEP_TIME: int64
    DEP_TIME: float64
    DEP_DELAY: float64
    DEP_DELAY_NEW: float64
    DEP_DEL15: float64
    DEP_TIME_BLK: object
    TAXI_OUT: float64
    TAXI_IN: float64
    CRS_ARR_TIME: int64
    ARR_TIME: float64
    ARR_DELAY: float64
    ARR_DELAY_NEW: float64
    ARR_DEL15: float64
    ARR_TIME_BLK: object
    CANCELLED: float64
    CANCELLATION_CODE: object
    DIVERTED: float64
    CRS_ELAPSED_TIME: float64
    A

## Save Merged Dataset

In [14]:
print("═" * 60)
print("FINAL MERGED DATASET AUDIT")
print("═" * 60)

print(f"\n── SHAPE ────────────────────────────────────────────────────")
print(f"  Rows: {flights.shape[0]:,}")
print(f"  Cols: {flights.shape[1]}")

print(f"\n── DATE RANGE ───────────────────────────────────────────────")
print(f"  Min: {flights['FL_DATE'].min().date()}")
print(f"  Max: {flights['FL_DATE'].max().date()}")
print(f"  Years: {sorted(flights['YEAR'].unique().tolist())}")
print(f"  Months in 2025: {sorted(flights[flights['YEAR']==2025]['MONTH'].unique().tolist())}")

print(f"\n── TARGET VARIABLE ──────────────────────────────────────────")
print(f"  ARR_DEL15 nulls: {flights['ARR_DEL15'].isna().sum():,}")
print(f"  ARR_DEL15 values: {flights['ARR_DEL15'].value_counts().to_dict()}")
print(f"  Delay rate: {flights['ARR_DEL15'].mean()*100:.2f}%")

print(f"\n── CARRIERS ─────────────────────────────────────────────────")
print(f"  Unique carriers: {sorted(flights['OP_UNIQUE_CARRIER'].unique().tolist())}")
print(f"  Count: {flights['OP_UNIQUE_CARRIER'].nunique()}")

print(f"\n── AIRPORTS ─────────────────────────────────────────────────")
print(f"  Unique origins: {flights['ORIGIN'].nunique()}")
print(f"  Unique dests: {flights['DEST'].nunique()}")

print(f"\n── EDA COLUMNS CARRIED THROUGH ──────────────────────────────")
eda_cols = ['AIRLINE_NAME', 'DAY_NAME', 'DIST_GROUP', 'TAXI_GROUP', 'AIRLINE', 'IS_HOLIDAY']
for col in eda_cols:
    status = "" if col in flights.columns else "MISSING"
    nulls = flights[col].isna().sum() if col in flights.columns else "N/A"
    print(f"  {status}  {col}  (nulls: {nulls})")

print(f"\n── WEATHER COLS ─────────────────────────────────────────────")
wx_cols = ['origin_temp','origin_wind_speed','origin_precip_1hr',
           'dest_temp','dest_wind_speed','dest_precip_1hr']
for col in wx_cols:
    pct = flights[col].isna().mean()*100
    print(f"  {col}: {pct:.1f}% null")

print(f"\n── FINANCIALS COLS ──────────────────────────────────────────")
fin_cols = ['OP_REVENUES','OP_EXPENSES','NET_INCOME','OP_PROFIT_LOSS','MAINTENANCE']
for col in fin_cols:
    pct = flights[col].isna().mean()*100
    print(f"  {col}: {pct:.1f}% null")

print(f"\n── T100 COLS ────────────────────────────────────────────────")
for col in ['origin_monthly_passengers', 'origin_monthly_departures']:
    pct = flights[col].isna().mean()*100
    print(f"  {col}: {pct:.1f}% null")

print(f"\n── DTYPES SANITY ────────────────────────────────────────────")
for col in flights.columns:
    print(f"  {col}: {flights[col].dtype}")

print(f"\n── DUPLICATE CHECK ──────────────────────────────────────────")
dupes = flights.duplicated(subset=['FL_DATE','OP_UNIQUE_CARRIER','OP_CARRIER_FL_NUM','ORIGIN','DEST']).sum()
print(f"  Duplicate flights: {dupes:,}")

print(f"\n── NULL SUMMARY (cols with >1% null) ────────────────────────")
null_counts = flights.isnull().sum()
null_counts = null_counts[(null_counts/len(flights)) > 0.01].sort_values(ascending=False)
for col, cnt in null_counts.items():
    pct = cnt/len(flights)*100
    print(f"  {col}: {cnt:,} ({pct:.1f}%)")

print("\n═" * 60)
print("AUDIT COMPLETE")
print("═" * 60)

════════════════════════════════════════════════════════════
FINAL MERGED DATASET AUDIT
════════════════════════════════════════════════════════════

── SHAPE ────────────────────────────────────────────────────
  Rows: 18,227,796
  Cols: 77

── DATE RANGE ───────────────────────────────────────────────
  Min: 2023-01-01
  Max: 2025-08-27
  Years: [2023, 2024, 2025]
  Months in 2025: [1, 2, 3, 4, 5, 6, 7, 8]

── TARGET VARIABLE ──────────────────────────────────────────
  ARR_DEL15 nulls: 0
  ARR_DEL15 values: {0.0: 14353289, 1.0: 3874507}
  Delay rate: 21.26%

── CARRIERS ─────────────────────────────────────────────────
  Unique carriers: ['9E', 'AA', 'AS', 'B6', 'DL', 'F9', 'G4', 'HA', 'MQ', 'NK', 'OH', 'OO', 'UA', 'WN', 'YX']
  Count: 15

── AIRPORTS ─────────────────────────────────────────────────
  Unique origins: 361
  Unique dests: 361

── EDA COLUMNS CARRIED THROUGH ──────────────────────────────
  MISSING  AIRLINE_NAME  (nulls: N/A)
  MISSING  DAY_NAME  (nulls: N/A)
  MISSIN